# 01 – Exploración y limpieza de datos (BCCR)

**Proyecto:** Periodismo de datos sobre la economía costarricense.

Este notebook carga, inspecciona y limpia las series de **PIB por régimen de comercio** publicadas por el Banco Central de Costa Rica (BCCR). Se generan dos CSVs limpios para análisis posterior:

- `bccr-pib-corrientes-limpio.csv` — PIB a precios corrientes (nominales), 1991-2026.
- `bccr-pib-encadenados-limpio.csv` — PIB encadenado (real), 1991-2026.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

## Carga de archivos BCCR

In [ ]:
data_raw = (Path("..") / "data" / "raw").resolve()

archivo_corrientes  = data_raw / "bccr-pib-regimen-corrientes-7720.xlsx"
archivo_encadenados = data_raw / "bccr-pib-regimen-encadenados-7721.xlsx"

raw_corrientes  = pd.read_excel(archivo_corrientes,  header=None, engine="openpyxl")
raw_encadenados = pd.read_excel(archivo_encadenados, header=None, engine="openpyxl")

pd.set_option("display.max_colwidth", None)

print("=== CORRIENTES (raw, header=None) ===")
print(raw_corrientes.to_string())

print("\n=== ENCADENADOS (raw, header=None) ===")
print(raw_encadenados.to_string())

## Limpieza: corrientes

In [ ]:
# Corrientes: todos los datos están en una sola columna de texto.
# Fila 3 (header=None): años separados por espacios.
# Filas 5, 6, 7: label + valores numéricos con coma decimal.

def _extraer_numeros(texto):
    """Extrae todos los valores numéricos con separador decimal coma o punto."""
    return [
        float(x.replace(",", "."))
        for x in re.findall(r"\d+[,.]\d+", str(texto))
    ]

year_text = str(raw_corrientes.iloc[3, 0])
years_corr = [int(t) for t in year_text.split() if t.isdigit() and len(t) == 4]
n = len(years_corr)

serie_total = _extraer_numeros(raw_corrientes.iloc[5, 0])[:n]
serie_def   = _extraer_numeros(raw_corrientes.iloc[6, 0])[:n]
serie_esp   = _extraer_numeros(raw_corrientes.iloc[7, 0])[:n]

corrientes_limpio = pd.DataFrame({
    "año":                years_corr,
    "pib_total":          serie_total,
    "regimen_definitivo": serie_def,
    "regimen_especial":   serie_esp,
})
corrientes_limpio["año"] = corrientes_limpio["año"].astype(int)
for col in ["pib_total", "regimen_definitivo", "regimen_especial"]:
    corrientes_limpio[col] = pd.to_numeric(corrientes_limpio[col], errors="coerce")

data_processed = (Path("..") / "data" / "processed").resolve()
data_processed.mkdir(parents=True, exist_ok=True)
corrientes_limpio.to_csv(
    data_processed / "bccr-pib-corrientes-limpio.csv",
    index=False, encoding="utf-8"
)
print(f"Guardado: {len(corrientes_limpio)} filas — {data_processed / 'bccr-pib-corrientes-limpio.csv'}")

## Limpieza: encadenados

In [ ]:
# Encadenados: formato multi-columna.
# Fila 4 (header=None), columnas 1..N: años (1991-2026).
# Filas 6, 7, 8: PIB total, régimen definitivo, régimen especial.

years_enc = raw_encadenados.iloc[4, 1:].dropna().astype(int).tolist()
n_enc     = len(years_enc)

serie_total_enc = raw_encadenados.iloc[6, 1:n_enc + 1].astype(float).tolist()
serie_def_enc   = raw_encadenados.iloc[7, 1:n_enc + 1].astype(float).tolist()
serie_esp_enc   = raw_encadenados.iloc[8, 1:n_enc + 1].astype(float).tolist()

encadenados_limpio = pd.DataFrame({
    "año":                years_enc,
    "pib_total":          serie_total_enc,
    "regimen_definitivo": serie_def_enc,
    "regimen_especial":   serie_esp_enc,
})
encadenados_limpio["año"] = encadenados_limpio["año"].astype(int)
for col in ["pib_total", "regimen_definitivo", "regimen_especial"]:
    encadenados_limpio[col] = pd.to_numeric(encadenados_limpio[col], errors="coerce")

encadenados_limpio.to_csv(
    data_processed / "bccr-pib-encadenados-limpio.csv",
    index=False, encoding="utf-8"
)
print(f"Guardado: {len(encadenados_limpio)} filas — {data_processed / 'bccr-pib-encadenados-limpio.csv'}")

## Verificación final

In [ ]:
print("=== corrientes_limpio.head(10) ===")
print(corrientes_limpio.head(10))
print()
corrientes_limpio.info()

print("\n=== encadenados_limpio.head(10) ===")
print(encadenados_limpio.head(10))
print()
encadenados_limpio.info()